In [1]:
import pandas as pd
import numpy as np

bench = pd.read_csv("stoxx600_sectors_history.csv", parse_dates=["Date"])
bench = bench.sort_values("Date").set_index("Date")

# on garde la colonne benchmark 
bench_col = "STOXX_600_Global"
bench = bench[[bench_col]].rename(columns={bench_col: "BENCH"})
bench = bench.dropna()

print(bench.shape)
print(bench.head())
print(bench.index.min(), bench.index.max())

(5479, 1)
                 BENCH
Date                  
2004-04-26  247.070007
2004-04-27  246.919998
2004-04-28  243.380005
2004-04-29  241.250000
2004-04-30  239.050003
2004-04-26 00:00:00 2026-02-09 00:00:00


In [3]:
xlsx_path = "cross_asset_sxxp600.xlsx"
xl = pd.ExcelFile(xlsx_path)

print("Sheets:", xl.sheet_names)

Sheets: ['SXNP', 'SX7P', 'SXDP', 'SXIP', 'S600ENP', 'SX8P', 'S600FOP', 'S600CPP', 'SX6P', 'SXFP', 'SXOP', 'SXKP', 'SXPP', 'S600PDP', 'SX4P', 'SXAP', 'SX86P', 'SXTP', 'SXRP', 'SXMP', 'DESCRIPTION']


In [5]:
xlsx_path = "cross_asset_sxxp600.xlsx"
xl = pd.ExcelFile(xlsx_path)

exclude = {"DESCRIPTION", "description", "Desc", "DESC"}
sector_sheets = [s for s in xl.sheet_names if s not in exclude]

def load_sector_sheet(xlsx_path: str, sheet_name: str) -> pd.Series:
    df = pd.read_excel(
        xlsx_path,
        sheet_name=sheet_name,
        skiprows=6  # on saute les 6 lignes Bloomberg
    )

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").set_index("Date")

    s = pd.to_numeric(df["PX_LAST"], errors="coerce").rename(sheet_name)

    return s

sector_series = [load_sector_sheet(xlsx_path, sh) for sh in sector_sheets]
sectors = pd.concat(sector_series, axis=1).sort_index()

print("sectors shape:", sectors.shape)
print("date range:", sectors.index.min(), "->", sectors.index.max())
print(sectors.tail())

sectors shape: (6719, 20)
date range: 1999-12-30 00:00:00 -> 2026-02-20 00:00:00
               SXNP    SX7P     SXDP    SXIP  S600ENP    SX8P  S600FOP  \
Date                                                                     
2026-02-16  1147.14  358.38  1216.86  488.39   152.37  830.51   199.46   
2026-02-17  1146.68  362.39  1234.01  492.95   151.39  837.21   197.76   
2026-02-18  1170.13  370.12  1233.49  493.63   154.77  860.76   197.34   
2026-02-19  1162.29  365.33  1224.52  493.01   156.15  855.70   200.88   
2026-02-20  1167.12  369.60  1225.42  498.16   155.23  855.65   200.89   

            S600CPP    SX6P    SXFP    SXOP    SXKP    SXPP  S600PDP     SX4P  \
Date                                                                            
2026-02-16   363.84  548.55  872.26  892.12  297.22  784.98   186.74  1199.11   
2026-02-17   367.72  550.68  870.89  891.16  299.34  772.46   186.26  1199.52   
2026-02-18   371.69  550.24  887.32  906.81  297.06  804.91   185.36  1188.0

In [7]:
start_dates = sectors.apply(lambda s: s.first_valid_index())
start_dates = start_dates.sort_values()

print(start_dates)
print("\nCommon start (all sectors available):", start_dates.max())
print("Earliest start (at least one sector):", start_dates.min())

SXNP      1999-12-30
SXTP      1999-12-30
SXAP      1999-12-30
SX4P      1999-12-30
SXPP      1999-12-30
SXKP      1999-12-30
SXOP      1999-12-30
SXRP      1999-12-30
SXFP      1999-12-30
SX8P      1999-12-30
SXIP      1999-12-30
SXDP      1999-12-30
SX7P      1999-12-30
SX6P      1999-12-30
SXMP      1999-12-30
SX86P     2000-12-29
S600CPP   2010-09-17
S600FOP   2010-09-17
S600ENP   2010-09-17
S600PDP   2010-09-17
dtype: datetime64[ns]

Common start (all sectors available): 2010-09-17 00:00:00
Earliest start (at least one sector): 1999-12-30 00:00:00


In [8]:
# prices = secteurs + BENCH (on le fera juste après si pas encore fait)
# Ici on part de sectors (prix daily)

sectors_m = sectors.resample("ME").last()
print("sectors_m shape:", sectors_m.shape)
print("range:", sectors_m.index.min(), "->", sectors_m.index.max())
print(sectors_m.tail())

sectors_m shape: (315, 20)
range: 1999-12-31 00:00:00 -> 2026-02-28 00:00:00
               SXNP    SX7P     SXDP    SXIP  S600ENP    SX8P  S600FOP  \
Date                                                                     
2025-10-31  1084.27  316.18  1075.78  478.95   132.44  869.63   183.75   
2025-11-30  1035.26  329.32  1129.72  492.58   134.98  827.00   190.26   
2025-12-31  1068.30  355.30  1142.11  510.39   134.57  836.60   187.57   
2026-01-31  1133.44  374.61  1179.39  486.52   146.87  880.82   186.15   
2026-02-28  1167.12  369.60  1225.42  498.16   155.23  855.65   200.89   

            S600CPP    SX6P    SXFP    SXOP    SXKP    SXPP  S600PDP     SX4P  \
Date                                                                            
2025-10-31   384.96  475.74  864.23  805.01  256.08  594.18   168.81  1126.87   
2025-11-30   391.39  485.75  844.82  834.93  252.04  603.88   167.13  1119.30   
2025-12-31   394.90  489.77  899.42  848.04  255.88  666.17   168.71  1107.47   

In [9]:
ret_m = sectors_m.pct_change()
print(ret_m.tail())

                SXNP      SX7P      SXDP      SXIP   S600ENP      SX8P  \
Date                                                                     
2025-10-31  0.016005  0.014015  0.044436 -0.016085  0.057405  0.034732   
2025-11-30 -0.045201  0.041559  0.050140  0.028458  0.019178 -0.049021   
2025-12-31  0.031915  0.078890  0.010967  0.036157 -0.003037  0.011608   
2026-01-31  0.060975  0.054348  0.032641 -0.046768  0.091402  0.052857   
2026-02-28  0.029715 -0.013374  0.039029  0.023925  0.056921 -0.028576   

             S600FOP   S600CPP      SX6P      SXFP      SXOP      SXKP  \
Date                                                                     
2025-10-31  0.023278  0.050712  0.074536  0.009579  0.010748  0.019346   
2025-11-30  0.035429  0.016703  0.021041 -0.022459  0.037167 -0.015776   
2025-12-31 -0.014139  0.008968  0.008276  0.064629  0.015702  0.015236   
2026-01-31 -0.007571 -0.075842  0.072708  0.007282  0.005778  0.042051   
2026-02-28  0.079183  0.027483  0.033

In [10]:
bench = pd.read_csv("stoxx600_sectors_history.csv", parse_dates=["Date"]).sort_values("Date").set_index("Date")
bench = bench[["STOXX_600_Global"]].rename(columns={"STOXX_600_Global":"BENCH"})
bench_m = bench.resample("ME").last()

bench_ret_m = bench_m.pct_change()

print("bench_m range:", bench_m.index.min(), "->", bench_m.index.max())
print(bench_m.tail())

bench_m range: 2004-04-30 00:00:00 -> 2026-02-28 00:00:00
                 BENCH
Date                  
2025-10-31  571.890015
2025-11-30  576.429993
2025-12-31  592.780029
2026-01-31  611.000000
2026-02-28  618.460022


In [11]:
# on joint sur les mêmes dates mensuelles
ret_all_m = ret_m.join(bench_ret_m, how="inner")

print("ret_all_m shape:", ret_all_m.shape)
print(ret_all_m.tail())

ret_all_m shape: (263, 21)
                SXNP      SX7P      SXDP      SXIP   S600ENP      SX8P  \
Date                                                                     
2025-10-31  0.016005  0.014015  0.044436 -0.016085  0.057405  0.034732   
2025-11-30 -0.045201  0.041559  0.050140  0.028458  0.019178 -0.049021   
2025-12-31  0.031915  0.078890  0.010967  0.036157 -0.003037  0.011608   
2026-01-31  0.060975  0.054348  0.032641 -0.046768  0.091402  0.052857   
2026-02-28  0.029715 -0.013374  0.039029  0.023925  0.056921 -0.028576   

             S600FOP   S600CPP      SX6P      SXFP  ...      SXKP      SXPP  \
Date                                                ...                       
2025-10-31  0.023278  0.050712  0.074536  0.009579  ...  0.019346  0.065240   
2025-11-30  0.035429  0.016703  0.021041 -0.022459  ... -0.015776  0.016325   
2025-12-31 -0.014139  0.008968  0.008276  0.064629  ...  0.015236  0.103150   
2026-01-31 -0.007571 -0.075842  0.072708  0.007282  ...  0.

In [12]:
print("Nb mois:", ret_all_m.shape[0])

Nb mois: 263


In [13]:
print("Common start:", pd.Timestamp("2010-09-17").to_period("M").to_timestamp("M"))

Common start: 2010-09-30 00:00:00


In [14]:
sectors_cols = [c for c in ret_all_m.columns if c != "BENCH"]

In [15]:
h = 6  # horizon en mois

# rendements futurs cumulés à horizon h
future_ret_6m = (1 + ret_all_m[sectors_cols]).rolling(h).apply(np.prod, raw=True).shift(-h) - 1

print(future_ret_6m.shape)
print(future_ret_6m.tail(10))

(263, 20)
                SXNP      SX7P      SXDP      SXIP   S600ENP      SX8P  \
Date                                                                     
2025-05-31  0.034918  0.194010  0.070733  0.021039  0.170279 -0.007751   
2025-06-30  0.055278  0.293646  0.116891  0.079711  0.123008 -0.009425   
2025-07-31  0.094117  0.272712  0.165820  0.000062  0.166468  0.078973   
2025-08-31  0.137156  0.238689  0.168837  0.020778  0.228863  0.087548   
2025-09-30       NaN       NaN       NaN       NaN       NaN       NaN   
2025-10-31       NaN       NaN       NaN       NaN       NaN       NaN   
2025-11-30       NaN       NaN       NaN       NaN       NaN       NaN   
2025-12-31       NaN       NaN       NaN       NaN       NaN       NaN   
2026-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2026-02-28       NaN       NaN       NaN       NaN       NaN       NaN   

             S600FOP   S600CPP      SX6P      SXFP      SXOP      SXKP  \
Date                       

In [16]:
future_bench_6m = (1 + ret_all_m["BENCH"]).rolling(h).apply(np.prod, raw=True).shift(-h) - 1

In [17]:
# Surperformance future (cible)
future_excess_6m = future_ret_6m.sub(future_bench_6m, axis=0)

print(future_excess_6m.shape)
print(future_excess_6m.tail(10))

(263, 20)
                SXNP      SX7P      SXDP      SXIP   S600ENP      SX8P  \
Date                                                                     
2025-05-31 -0.015677  0.143415  0.020138 -0.029556  0.119684 -0.058346   
2025-06-30 -0.039685  0.198684  0.021928 -0.015252  0.028045 -0.104388   
2025-07-31 -0.024705  0.153890  0.046998 -0.118761  0.047646 -0.039849   
2025-08-31  0.012969  0.114502  0.044650 -0.103409  0.104677 -0.036639   
2025-09-30       NaN       NaN       NaN       NaN       NaN       NaN   
2025-10-31       NaN       NaN       NaN       NaN       NaN       NaN   
2025-11-30       NaN       NaN       NaN       NaN       NaN       NaN   
2025-12-31       NaN       NaN       NaN       NaN       NaN       NaN   
2026-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2026-02-28       NaN       NaN       NaN       NaN       NaN       NaN   

             S600FOP   S600CPP      SX6P      SXFP      SXOP      SXKP  \
Date                       

In [18]:
print("NA ratio:", future_excess_6m.isna().mean().mean())
print("first non-NA date:", future_excess_6m.dropna(how="all").index.min())
print("last non-NA date :", future_excess_6m.dropna(how="all").index.max())

NA ratio: 0.08136882129277566
first non-NA date: 2004-04-30 00:00:00
last non-NA date : 2025-08-31 00:00:00


In [19]:
pmi_manu = pd.read_csv("pmmneu_m_d.csv", parse_dates=["Date"])
pmi_serv = pd.read_csv("pmsreu_m_d.csv", parse_dates=["Date"])

pmi_manu = pmi_manu.sort_values("Date").set_index("Date")[["Close"]].rename(columns={"Close":"PMI_M"})
pmi_serv = pmi_serv.sort_values("Date").set_index("Date")[["Close"]].rename(columns={"Close":"PMI_S"})

pmi = pmi_manu.join(pmi_serv, how="inner")
pmi["PMI"] = pmi.mean(axis=1)

print(pmi.head())
print(pmi.index.min(), "->", pmi.index.max())

            PMI_M  PMI_S    PMI
Date                           
2010-06-30   55.6   55.5  55.55
2010-07-30   56.7   55.8  56.25
2010-08-31   55.1   55.9  55.50
2010-09-30   53.7   54.1  53.90
2010-10-29   54.6   53.3  53.95
2010-06-30 00:00:00 -> 2026-01-30 00:00:00


In [20]:
pmi_m = pmi["PMI"].resample("ME").last()

print(pmi_m.head())

Date
2010-06-30    55.55
2010-07-31    56.25
2010-08-31    55.50
2010-09-30    53.90
2010-10-31    53.95
Freq: ME, Name: PMI, dtype: float64


In [21]:
# Z score rolling sur 3 ans
window = 36

pmi_z = (pmi_m - pmi_m.rolling(window).mean()) / pmi_m.rolling(window).std()

print(pmi_z.tail())

Date
2025-09-30    1.181116
2025-10-31    1.670530
2025-11-30    1.612385
2025-12-31    0.960030
2026-01-31    0.909354
Freq: ME, Name: PMI, dtype: float64


In [22]:
# Aligner avec cible
data_test = pd.concat([pmi_z, future_excess_6m], axis=1).dropna()

print(data_test.shape)

(112, 21)


In [23]:
# Corr entre PMI et excess return pour chaque secteur 
corr_pmi = data_test[sectors_cols].corrwith(data_test["PMI"])

corr_pmi = corr_pmi.sort_values(ascending=False)

print(corr_pmi)

S600FOP    0.566721
SXDP       0.344972
SX6P       0.282138
S600PDP    0.275534
S600CPP    0.203344
SX4P       0.175012
SX86P      0.136062
S600ENP    0.069445
SXPP       0.052362
SXKP       0.005200
SXAP      -0.071334
SX8P      -0.072861
SXTP      -0.085228
SXFP      -0.134719
SXIP      -0.143963
SXMP      -0.250895
SXOP      -0.358194
SXRP      -0.375021
SX7P      -0.382498
SXNP      -0.390152
dtype: float64


In [24]:
# Variation PMI 
pmi_change = pmi_m.diff()

# standardisation
pmi_change_z = (pmi_change - pmi_change.rolling(36).mean()) / pmi_change.rolling(36).std()

data_test2 = pd.concat([pmi_change_z, future_excess_6m], axis=1).dropna()

corr_pmi_change = data_test2[sectors_cols].corrwith(data_test2["PMI"])

corr_pmi_change = corr_pmi_change.sort_values(ascending=False)

print(corr_pmi_change)

SX6P       0.172709
SX7P       0.134055
SXKP       0.129457
SXDP       0.078815
SXFP       0.056118
SXNP       0.000464
SXTP      -0.015582
S600PDP   -0.027230
SX86P     -0.029555
SXRP      -0.055975
SXOP      -0.097298
SX8P      -0.103757
SXIP      -0.104970
S600FOP   -0.116624
S600ENP   -0.135373
SXPP      -0.155636
S600CPP   -0.158822
SXMP      -0.193982
SXAP      -0.206409
SX4P      -0.242869
dtype: float64


Comparé au niveau du PMI :

Le niveau donnait des corrélations fortes mais contre-intuitives.

La variation donne des corrélations plus faibles mais plus cohérentes.

Exemple : Banks passent de fortement négatif (niveau) à positif (variation).
Ça fait sens car accélération du cycle = hausse crédit = bénéfique aux banques.

Mais : Les corrélations restent modestes (0.10–0.20).
Donc PMI seul ne suffit PAS pour une vraie rotation sectorielle robuste.

In [ ]:
def find_cell(df: pd.DataFrame, text: str):
    mask = df.astype(str).apply(lambda col: col.str.contains(text, na=False))
    if not mask.any().any():
        return None
    r, c = np.where(mask.values)
    return int(r[0]), int(c[0])  # indices numériques

def extract_block_debug(df: pd.DataFrame, label: str, start_offset_rows=1, value_col_offset=1, n_preview=12):
    """
    Extrait un bloc sous un label et affiche un debug complet.
    """
    pos = find_cell(df, label)
    if pos is None:
        raise ValueError(f"Label introuvable: {label}")
    r0, c0 = pos

    col_date = df.columns[c0]
    col_val  = df.columns[c0 + value_col_offset]

    raw = df.iloc[r0 + start_offset_rows : r0 + start_offset_rows + n_preview, [c0, c0 + value_col_offset]].copy()
    raw.columns = ["raw_date", "raw_value"]

    # conversion
    tmp = df.iloc[r0 + start_offset_rows :, [c0, c0 + value_col_offset]].copy()
    tmp.columns = ["Date", "Value"]
    tmp["Date"]  = pd.to_datetime(tmp["Date"], errors="coerce")
    tmp["Value"] = pd.to_numeric(tmp["Value"], errors="coerce")
    tmp = tmp.dropna(subset=["Date", "Value"]).set_index("Date").sort_index()

    # stats
    date_diffs = tmp.index.to_series().diff().dropna()
    if len(date_diffs) > 0:
        freq_guess = date_diffs.value_counts().head(3)
    else:
        freq_guess = "NA"

    print("\n==============================")
    print("LABEL:", label)
    print("Found at (row, col_index):", (r0, c0))
    print("Using columns:", col_date, "and", col_val)
    print("\nRAW PREVIEW (just below label):")
    print(raw)
    print("\nCLEAN STATS:")
    print("n =", len(tmp))
    print("date min/max:", tmp.index.min(), "->", tmp.index.max())
    print("value min/max:", tmp["Value"].min(), "->", tmp["Value"].max())
    print("freq guess:", freq_guess)
    print("==============================\n")

    return tmp["Value"]

In [35]:
cross = pd.read_excel("cross_asset.xlsx")

In [36]:
bund2y = extract_block_debug(cross, "2Y Bund nominal", start_offset_rows=1, value_col_offset=1)


LABEL: 2Y Bund nominal
Found at (row, col_index): (4, 17)
Using columns: Unnamed: 17 and Unnamed: 18

RAW PREVIEW (just below label):
               raw_date  raw_value
5                   NaN        NaN
6   2010-01-01 00:00:00      1.332
7   2010-01-04 00:00:00      1.347
8   2010-01-05 00:00:00      1.286
9   2010-01-06 00:00:00      1.294
10  2010-01-07 00:00:00      1.264
11  2010-01-08 00:00:00      1.244
12  2010-01-11 00:00:00      1.225
13  2010-01-12 00:00:00      1.210
14  2010-01-13 00:00:00      1.198
15  2010-01-14 00:00:00      1.179
16  2010-01-15 00:00:00      1.135

CLEAN STATS:
n = 4197
date min/max: 2010-01-01 00:00:00 -> 2026-02-13 00:00:00
value min/max: -1.003 -> 3.336
freq guess: Date
1 days    3349
3 days     838
2 days       6
Name: count, dtype: int64



In [37]:
bund10y = extract_block_debug(cross, "10Y Bund nominal", start_offset_rows=1, value_col_offset=1)


LABEL: 10Y Bund nominal
Found at (row, col_index): (4, 20)
Using columns: Unnamed: 20 and Unnamed: 21

RAW PREVIEW (just below label):
               raw_date  raw_value
5                   NaN        NaN
6   2010-01-01 00:00:00      3.387
7   2010-01-04 00:00:00      3.391
8   2010-01-05 00:00:00      3.373
9   2010-01-06 00:00:00      3.381
10  2010-01-07 00:00:00      3.370
11  2010-01-08 00:00:00      3.385
12  2010-01-11 00:00:00      3.347
13  2010-01-12 00:00:00      3.313
14  2010-01-13 00:00:00      3.303
15  2010-01-14 00:00:00      3.298
16  2010-01-15 00:00:00      3.262

CLEAN STATS:
n = 4198
date min/max: 2010-01-01 00:00:00 -> 2026-02-13 00:00:00
value min/max: -0.856 -> 3.491
freq guess: Date
1 days    3351
3 days     838
2 days       5
Name: count, dtype: int64



In [38]:
# Pente daily
rates = pd.concat(
    [bund2y.rename("BUND_2Y"), bund10y.rename("BUND_10Y")],
    axis=1,
    join="inner"   # garde uniquement dates communes
).dropna()

rates["SLOPE_NOM"] = rates["BUND_10Y"] - rates["BUND_2Y"]
print(rates.head())
print(rates.tail())
print("Slope min/max:", rates["SLOPE_NOM"].min(), rates["SLOPE_NOM"].max())

            BUND_2Y  BUND_10Y  SLOPE_NOM
Date                                    
2010-01-01    1.332     3.387      2.055
2010-01-04    1.347     3.391      2.044
2010-01-05    1.286     3.373      2.087
2010-01-06    1.294     3.381      2.087
2010-01-07    1.264     3.370      2.106
            BUND_2Y  BUND_10Y  SLOPE_NOM
Date                                    
2026-02-09    2.078     2.840      0.762
2026-02-10    2.069     2.808      0.739
2026-02-11    2.066     2.792      0.726
2026-02-12    2.060     2.779      0.719
2026-02-13    2.036     2.755      0.719
Slope min/max: -0.823 2.3579999999999997


In [40]:
slope_m = rates["SLOPE_NOM"].resample("ME").last()

print(slope_m.head(10))
print(slope_m.tail(10))
print("Monthly slope min/max:", slope_m.min(), slope_m.max())

Date
2010-01-31    2.077
2010-02-28    2.146
2010-03-31    2.134
2010-04-30    2.246
2010-05-31    2.146
2010-06-30    1.977
2010-07-31    1.885
2010-08-31    1.526
2010-09-30    1.444
2010-10-31    1.529
Freq: ME, Name: SLOPE_NOM, dtype: float64
Date
2025-05-31    0.724
2025-06-30    0.746
2025-07-31    0.731
2025-08-31    0.784
2025-09-30    0.692
2025-10-31    0.665
2025-11-30    0.660
2025-12-31    0.733
2026-01-31    0.754
2026-02-28    0.719
Freq: ME, Name: SLOPE_NOM, dtype: float64
Monthly slope min/max: -0.8040000000000003 2.246


In [41]:
# Z score
window = 36
slope_z = (slope_m - slope_m.rolling(window).mean()) / slope_m.rolling(window).std()

print(slope_z.dropna().head())
print(slope_z.dropna().tail())

Date
2012-12-31   -1.075088
2013-01-31   -0.752018
2013-02-28   -0.700999
2013-03-31   -1.045192
2013-04-30   -1.469167
Freq: ME, Name: SLOPE_NOM, dtype: float64
Date
2025-10-31    1.444169
2025-11-30    1.349489
2025-12-31    1.404283
2026-01-31    1.357397
2026-02-28    1.218055
Freq: ME, Name: SLOPE_NOM, dtype: float64


In [27]:
cross = pd.read_excel("cross_asset.xlsx")

In [28]:
bund2y = extract_series_under_label(cross, "2Y Bund nominal")
bund10y = extract_series_under_label(cross, "10Y Bund nominal")

print(bund2y.head(), "\n---\n", bund2y.tail())
print("\n====\n")
print(bund10y.head(), "\n---\n", bund10y.tail())

Date
2010-01-01    1.332
2010-01-04    1.347
2010-01-05    1.286
2010-01-06    1.294
2010-01-07    1.264
Name: Value, dtype: float64 
---
 Date
2026-02-09    2.078
2026-02-10    2.069
2026-02-11    2.066
2026-02-12    2.060
2026-02-13    2.036
Name: Value, dtype: float64

====

Date
2010-01-01    3.387
2010-01-04    3.391
2010-01-05    3.373
2010-01-06    3.381
2010-01-07    3.370
Name: Value, dtype: float64 
---
 Date
2026-02-09    2.840
2026-02-10    2.808
2026-02-11    2.792
2026-02-12    2.779
2026-02-13    2.755
Name: Value, dtype: float64


In [29]:
rates = pd.concat([bund2y.rename("BUND_2Y"), bund10y.rename("BUND_10Y")], axis=1).dropna()
rates["SLOPE_NOM"] = rates["BUND_10Y"] - rates["BUND_2Y"]

slope_m = rates["SLOPE_NOM"].resample("ME").last()

window = 36
slope_z = (slope_m - slope_m.rolling(window).mean()) / slope_m.rolling(window).std()

print(slope_z.dropna().head())
print(slope_z.dropna().tail())

Date
2012-12-31   -1.075088
2013-01-31   -0.752018
2013-02-28   -0.700999
2013-03-31   -1.045192
2013-04-30   -1.469167
Freq: ME, Name: SLOPE_NOM, dtype: float64
Date
2025-10-31    1.444169
2025-11-30    1.349489
2025-12-31    1.404283
2026-01-31    1.357397
2026-02-28    1.218055
Freq: ME, Name: SLOPE_NOM, dtype: float64


In [43]:
def univariate_corr(factor: pd.Series, factor_name: str):
    tmp = pd.concat([factor.rename(factor_name), future_excess_6m], axis=1, join="inner").dropna()
    corr = tmp[sectors_cols].corrwith(tmp[factor_name]).sort_values(ascending=False)
    return corr, tmp

In [46]:
# Test slope
corr_slope, tmp_slope = univariate_corr(slope_z, "SLOPE_NOM")
print("SLOPE_Z corr top/bottom:")
print(corr_slope.head(5))
print(corr_slope.tail(5))
print("n obs:", tmp_slope.shape[0])

SLOPE_Z corr top/bottom:
S600ENP    0.507898
SXPP       0.279291
SX6P       0.225981
SX7P       0.153814
S600FOP    0.058962
dtype: float64
SX8P   -0.326603
SXMP   -0.345156
SX4P   -0.360164
SXRP   -0.403936
SXFP   -0.514758
dtype: float64
n obs: 153


In [47]:
# Variation de pente
slope_change = slope_m.diff()

slope_change_z = (slope_change - slope_change.rolling(36).mean()) / slope_change.rolling(36).std()

data_slope_change = pd.concat([slope_change_z.rename("SLOPE_CHG_Z"), future_excess_6m], axis=1).dropna()

corr_slope_change = data_slope_change[sectors_cols].corrwith(data_slope_change["SLOPE_CHG_Z"])
corr_slope_change = corr_slope_change.sort_values(ascending=False)

print(corr_slope_change)

SXKP       0.092496
SXNP       0.058042
SX86P      0.052503
S600FOP    0.049382
SX7P       0.042875
SXFP       0.028599
SXIP       0.010743
S600ENP    0.007131
SXMP      -0.006862
S600PDP   -0.014015
SX6P      -0.017822
SXOP      -0.028845
SXPP      -0.031195
SX8P      -0.035967
SXTP      -0.040720
SXAP      -0.062450
SXDP      -0.075642
S600CPP   -0.124153
SX4P      -0.155409
SXRP      -0.222036
dtype: float64


La variation de pente est beaucoup moins prédictive que le niveau de pente.

In [ ]:
# Autocorr PMI Slope
aligned = pd.concat(
    [pmi_z.rename("PMI_Z"), slope_z.rename("SLOPE_Z")],
    axis=1,
    join="inner"
).dropna()

print(aligned.corr())
print("n aligned:", len(aligned))

            PMI_Z   SLOPE_Z
PMI_Z    1.000000  0.322693
SLOPE_Z  0.322693  1.000000
n aligned: 117


Les deux facteurs capturent des choses différentes
On peut les combiner sans trop de double comptage

# Spread

In [52]:
ig = extract_block_debug(cross, "IG Europe OAS", start_offset_rows=1, value_col_offset=1)


LABEL: IG Europe OAS
Found at (row, col_index): (4, 44)
Using columns: Unnamed: 44 and Unnamed: 45

RAW PREVIEW (just below label):
               raw_date  raw_value
5         LECPOAS Index        NaN
6   2010-01-04 00:00:00       1.72
7   2010-01-05 00:00:00       1.70
8   2010-01-06 00:00:00       1.67
9   2010-01-07 00:00:00       1.64
10  2010-01-08 00:00:00       1.61
11  2010-01-11 00:00:00       1.58
12  2010-01-12 00:00:00       1.57
13  2010-01-13 00:00:00       1.58
14  2010-01-14 00:00:00       1.59
15  2010-01-15 00:00:00       1.60
16  2010-01-18 00:00:00       1.59

CLEAN STATS:
n = 4105
date min/max: 1970-01-01 00:00:00.000046066 -> 2026-02-12 00:00:00
value min/max: 0.72 -> 3.77
freq guess: Date
1 days    3251
3 days     783
4 days      38
Name: count, dtype: int64



C:\Users\Lisa Benkhaled\AppData\Local\Temp\ipykernel_16076\1615307219.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tmp["Date"]  = pd.to_datetime(tmp["Date"], errors="coerce")


In [53]:
print("min date:", ig.index.min())
print("rows with year 1970:", ig[ig.index.year == 1970].head(20))
print("count year 1970:", (ig.index.year == 1970).sum())

min date: 1970-01-01 00:00:00.000046066
rows with year 1970: Date
1970-01-01 00:00:00.000046066    0.77
Name: Value, dtype: float64
count year 1970: 1


In [54]:
ig_clean = ig[ig.index >= "2000-01-01"]
print("clean min/max:", ig_clean.index.min(), "->", ig_clean.index.max())
print("removed:", len(ig) - len(ig_clean))

clean min/max: 2010-01-04 00:00:00 -> 2026-02-12 00:00:00
removed: 1


In [51]:
hy = extract_block_debug(cross, "HY Europe OAS", start_offset_rows=1, value_col_offset=1)


LABEL: HY Europe OAS
Found at (row, col_index): (4, 48)
Using columns: Unnamed: 48 and Unnamed: 49

RAW PREVIEW (just below label):
               raw_date  raw_value
5        LP01OAS  Index        NaN
6   2010-01-04 00:00:00       7.33
7   2010-01-05 00:00:00       7.18
8   2010-01-06 00:00:00       7.02
9   2010-01-07 00:00:00       6.91
10  2010-01-08 00:00:00       6.80
11  2010-01-11 00:00:00       6.72
12  2010-01-12 00:00:00       6.71
13  2010-01-13 00:00:00       6.76
14  2010-01-14 00:00:00       6.75
15  2010-01-15 00:00:00       6.75
16  2010-01-18 00:00:00       6.78

CLEAN STATS:
n = 4104
date min/max: 2010-01-04 00:00:00 -> 2026-02-13 00:00:00
value min/max: 2.38 -> 10.54
freq guess: Date
1 days    3252
3 days     781
4 days      40
Name: count, dtype: int64



C:\Users\Lisa Benkhaled\AppData\Local\Temp\ipykernel_16076\1615307219.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tmp["Date"]  = pd.to_datetime(tmp["Date"], errors="coerce")


In [56]:
hy_clean = hy[hy.index >= "2000-01-01"]

In [57]:
# Spread 
credit = pd.concat([ig_clean.rename("IG"), hy_clean.rename("HY")], axis=1, join="inner").dropna()
credit["SPREAD"] = credit["HY"] - credit["IG"]

# sanity checks
print(credit[["IG","HY","SPREAD"]].head())
print("spread min/max:", credit["SPREAD"].min(), credit["SPREAD"].max())

# mensuel
spread_m = credit["SPREAD"].resample("ME").last()

# z-score 36 mois
spread_z = (spread_m - spread_m.rolling(36).mean()) / spread_m.rolling(36).std()

print(spread_z.dropna().head())
print(spread_z.dropna().tail())

              IG    HY  SPREAD
Date                          
2010-01-04  1.72  7.33    5.61
2010-01-05  1.70  7.18    5.48
2010-01-06  1.67  7.02    5.35
2010-01-07  1.64  6.91    5.27
2010-01-08  1.61  6.80    5.19
spread min/max: 1.5099999999999998 6.77
Date
2012-12-31   -1.366913
2013-01-31   -1.434048
2013-02-28   -1.481083
2013-03-31   -1.323648
2013-04-30   -1.579309
Freq: ME, Name: SPREAD, dtype: float64
Date
2025-10-31   -1.098512
2025-11-30   -1.252713
2025-12-31   -1.257027
2026-01-31   -1.281819
2026-02-28   -1.050476
Freq: ME, Name: SPREAD, dtype: float64


In [58]:
corr_spread, tmp_spread = univariate_corr(spread_z, "SPREAD_Z")
print("SPREAD_Z corr top/bottom:")
print(corr_spread.head(5))
print(corr_spread.tail(5))
print("n obs:", tmp_spread.shape[0])

SPREAD_Z corr top/bottom:
SXNP       0.454052
S600CPP    0.412906
SXOP       0.346364
SX4P       0.334341
SX8P       0.324307
dtype: float64
SX6P    -0.145333
SXDP    -0.177858
SXIP    -0.233268
SX86P   -0.271748
SXKP    -0.451801
dtype: float64
n obs: 153


C’est contre-intuitif pour les cycliques (Industrials 0.45)

Ça suggère :

- soit effet mean-reversion
- soit timing du z-score (rolling 36m)
- soit fenêtre 6M spécifique

In [59]:
aligned_all = pd.concat(
    [pmi_z.rename("PMI_Z"),
     slope_z.rename("SLOPE_Z"),
     spread_z.rename("SPREAD_Z")],
    axis=1,
    join="inner"
).dropna()

print(aligned_all.corr())
print("n aligned:", len(aligned_all))

             PMI_Z   SLOPE_Z  SPREAD_Z
PMI_Z     1.000000  0.322693 -0.332720
SLOPE_Z   0.322693  1.000000 -0.298791
SPREAD_Z -0.332720 -0.298791  1.000000
n aligned: 117


In [ ]:
# Est ce qu'un facteur domine les autres
def mean_abs_corr(corr_series):
    return np.mean(np.abs(corr_series))

print("PMI mean abs corr:", mean_abs_corr(corr_pmi))
print("SLOPE mean abs corr:", mean_abs_corr(corr_slope))
print("SPREAD mean abs corr:", mean_abs_corr(corr_spread))

PMI mean abs corr: 0.21878271187296097
SLOPE mean abs corr: 0.2378737785782355
SPREAD mean abs corr: 0.2155592253841981


Aucun facteur n’écrase les autres